# MovieLens EDA: Ratings, Genres & a Fair Top-10
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/02_Data_Analysis_EDA/movies_ratings_eda.ipynb)

Exploratory analysis on the MovieLens small dataset (~100k ratings): what do people watch, how do genres compare, and which films are genuinely BEST once popularity is handled honestly?

Data lives in `../content/` - no downloads needed.

## 1. Load + merge

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

movies = pd.read_csv("../content/movies.csv")
ratings = pd.read_csv("../content/ratings.csv")
print(movies.shape, ratings.shape)

df = ratings.merge(movies, on="movieId")
df.head(3)

## 2. Rating behavior

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df.rating.value_counts().sort_index().plot.bar(ax=axes[0], title="rating distribution")
df.groupby("userId").size().hist(bins=40, ax=axes[1])
axes[1].set_title("ratings per user (long tail)")
df.groupby("movieId").size().hist(bins=40, ax=axes[2])
axes[2].set_title("ratings per movie (long tail)")
plt.tight_layout(); plt.show()

print("median ratings/user:", df.groupby('userId').size().median())
print("movies with <5 ratings:", round((df.groupby('movieId').size() < 5).mean(), 2), "%")

## 3. The naive Top-10 is wrong

In [ ]:
naive = df.groupby(["movieId", "title"]).rating \
          .agg(["mean", "count"]).reset_index()
print("NAIVE top-10 (tiny-sample flukes):")
print(naive.sort_values("mean", ascending=False).head(10)[["title", "mean", "count"]]
      .to_string(index=False))

One 5-star rating beats The Godfather. Fix: **Bayesian weighted rating** shrinks low-count means toward the global mean.

In [ ]:
C = df.rating.mean()                       # prior = global mean
m = 50                                     # shrinkage strength (min votes)

stats_df = df.groupby(["movieId", "title"]).rating.agg(["mean", "count"]).reset_index()
stats_df["score"] = ((stats_df["count"] / (stats_df["count"] + m)) * stats_df["mean"]
                     + (m / (stats_df["count"] + m)) * C)

print("FAIR top-10:")
print(stats_df.sort_values("score", ascending=False).head(10)
      [["title", "mean", "count", "score"]].to_string(index=False,
      formatters={"mean": "{:.2f}".format, "score": "{:.2f}".format}))

## 4. Genre landscape

In [ ]:
genres = (df.assign(genre=df.genres.str.split("|"))
            .explode("genre"))
top_genres = genres.genre.value_counts().head(10)
top_genres.plot.barh(figsize=(8, 4), title="ratings volume by genre")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

genre_quality = (genres.groupby("genre").rating
                 .agg(["mean", "count"]).query("count > 3000")
                 .sort_values("mean"))
genre_quality["mean"].plot.barh(figsize=(8, 5), color="teal",
                                title="avg rating by genre (min 3k ratings)")
plt.tight_layout(); plt.show()

## 5. Time dimension: decades

In [ ]:
df["year"] = df.title.str.extract(r"\((\d{4})\)").astype(float)
by_decade = (df.dropna(subset="year")
               .assign(decade=(df.year // 10 * 10).astype(int))
               .groupby("decade")
               .agg(avg_rating=("rating", "mean"), titles=("movieId", "nunique")))
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(by_decade.index, by_decade.titles, alpha=.35, label="#titles rated")
ax2 = ax1.twinx()
ax2.plot(by_decade.index, by_decade.avg_rating, "o-", color="crimson", label="avg rating")
ax1.set_xlabel("decade"); ax1.set_ylabel("titles"); ax2.set_ylabel("avg rating")
plt.title("Catalog coverage vs quality across film history"); plt.show()

## Takeaways
- Long tails everywhere: medians beat means when describing user activity.
- **Never rank by raw average** - shrinkage (IMDb's own formula) is a 3-line fix.
- Genre quality differences are real but modest; volume differences are huge.
- Next step: feed these cleaned features into the recommender notebooks in `03_Machine_Learning/projects/other/`.